# Eidos Colab GPU Bridge

Use this notebook only for Eidos Brain/Sentinel proof work that needs a Colab GPU. It runs existing repo-root proof commands through `tools/colab_gpu_bridge.py`, records GPU and Drive receipts, and leaves core Eidos behavior unchanged.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/bmparent/eidos.git"
BRANCH = "main"
PROOF_ROOT = Path("/content/eidos")

MODE = "proof-baseline"  # tensor-smoke, proof-baseline, or labeled-domain
SUITE = "smoke"
SEED = 42
FRAMES = 10000
RUN_ID = f"colab_gpu_{MODE}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
OUT = f"artifacts/proof_runs/{datetime.now(timezone.utc).strftime('%Y-%m-%d')}/{RUN_ID}"

# Labeled-domain settings are ignored unless MODE = "labeled-domain".
LABELED_FILE = "artifacts/cicids_webattacks_samples/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
LABEL_COLUMN = "Label"
ATTACK_LABELS = [
    "Web Attack - Brute Force",
    "Web Attack - XSS",
    "Web Attack - Sql Injection",
]


In [ ]:
def run(cmd, cwd=None):
    print("$", " ".join(str(part) for part in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

if not PROOF_ROOT.exists():
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROOF_ROOT)])
else:
    run(["git", "fetch", "origin"], cwd=PROOF_ROOT)
    run(["git", "checkout", BRANCH], cwd=PROOF_ROOT)
    run(["git", "pull", "--ff-only"], cwd=PROOF_ROOT)

os.chdir(PROOF_ROOT)
print("proof_root=", Path.cwd())
run(["git", "status", "--short", "--branch"], cwd=PROOF_ROOT)


In [ ]:
# Keep dependency setup conservative. Most Colab GPU runtimes already include torch.
if Path("requirements.txt").exists():
    run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=PROOF_ROOT)
elif Path("pyproject.toml").exists():
    run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=PROOF_ROOT)
else:
    run([sys.executable, "-m", "pip", "install", "numpy", "pytest"], cwd=PROOF_ROOT)


In [ ]:
cmd = [
    sys.executable,
    "tools/colab_gpu_bridge.py",
    "--mode", MODE,
    "--suite", SUITE,
    "--seed", str(SEED),
    "--frames", str(FRAMES),
    "--out", OUT,
    "--mount-drive",
    "--require-cuda",
]

if MODE == "labeled-domain":
    cmd.extend([
        "--dataset", "cicids_webattacks",
        "--file", LABELED_FILE,
        "--label-column", LABEL_COLUMN,
        "--sample-mode", "natural",
        "--confirmation-mode", "balanced",
        "--calibration-enabled",
    ])
    for label in ATTACK_LABELS:
        cmd.extend(["--attack-labels", label])

run(cmd, cwd=PROOF_ROOT)


In [ ]:
artifact_dir = PROOF_ROOT / OUT
print("artifact_dir=", artifact_dir)
for name in [
    "colab_gpu_bridge_receipt.md",
    "proof_digest.md",
    "benchmark_summary.md",
    "drive_manifest.json",
    "run_manifest.json",
]:
    path = artifact_dir / name
    print(f"{name}: {'present' if path.exists() else 'missing'}")
    if path.exists() and path.suffix == ".md":
        print(path.read_text(encoding="utf-8")[:2000])
